### 06.1 — Outlier Treatment Experiments

We compare three approaches:

1. **Baseline:** Retain all original training observations.
2. **Outlier removal:** Remove selected extreme observations from training only.
3. **Outlier capping:** Limit selected extreme feature values using training-derived thresholds.

All approaches will use the same XGBoost configuration and preprocessing methodology.

We retain the chronological split:
- Training: January 2013 – September 2015
- Validation: October 2015 – December 2015

The 2016 test set remains excluded from experiment selection. Our primary comparison metric is PR-AUC, supported by ROC-AUC, F1, precision, and recall.

In [1]:

from pathlib import Path
from datetime import date
import sys
import numpy as np
import pandas as pd
import polars as pl

from IPython.display import display

# Project paths
ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from loanrisk.predict import load_model

SOURCE = ROOT / "data/interim/lendingclub_2013_2016_36m_raw.parquet"
REPORT_DIR = ROOT / "reports/outlier_experiments"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Frozen model: reference only, no modification
bundle = load_model()
raw_features = list(bundle["raw_columns"])

# Chronological development splits
lf = pl.scan_parquet(SOURCE)

train_lf = lf.filter(
    (pl.col("issue_date") >= date(2013, 1, 1))
    & (pl.col("issue_date") < date(2015, 10, 1))
)

val_lf = lf.filter(
    (pl.col("issue_date") >= date(2015, 10, 1))
    & (pl.col("issue_date") < date(2016, 1, 1))
)

# Check sizes and default rates without loading full datasets
def summarize(data, name):
    row = data.select(
        pl.len().alias("rows"),
        pl.col("target").cast(pl.Int8).sum().alias("defaults")
    ).collect().row(0, named=True)

    return {
        "split": name,
        "rows": row["rows"],
        "defaults": row["defaults"],
        "default_rate": row["defaults"] / row["rows"],
    }

split_summary = pd.DataFrame([
    summarize(train_lf, "Training"),
    summarize(val_lf, "Validation"),
])

assert split_summary.loc[0, "rows"] == 457355
assert split_summary.loc[1, "rows"] == 88810
assert len(raw_features) == 55

print("Saved model:", bundle["model_name"])
print("Raw features:", len(raw_features))
print("Decision threshold:", round(float(bundle["threshold"]), 5))

print("\nDevelopment data:")
display(split_summary.round(4))

print("\nSetup: PASS")
print("2016 test set not loaded.")


Saved model: xgboost
Raw features: 55
Decision threshold: 0.1561

Development data:


,split,rows,defaults,default_rate
0,Training,457355,63688,0.1393
1,Validation,88810,13183,0.1484



Setup: PASS
2016 test set not loaded.



### 06.2 — Training-Only Outlier Impact

The EDA identified several strongly right-skewed continuous financial features. We select five for controlled outlier experiments.

Before modifying the training data, we measure how many unique loans are flagged, how much of the training set would be removed, and whether the flagged records have a different default rate.

IQR limits are derived exclusively from training data. No observations are removed at this stage.


In [2]:

# 06.2 — Combined outlier impact

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

outlier_report = pd.read_csv(
    ROOT / "reports/eda/01_training_outliers.csv"
)

selected_features = [
    "annual_inc",
    "revol_bal",
    "total_rev_hi_lim",
    "avg_cur_bal",
    "tot_cur_bal",
]

limits = (
    outlier_report
    .set_index("feature")
    .loc[selected_features]
)

# Load only the selected training columns
expressions = []

for col in selected_features:
    value = (
        pl.col(col)
        .cast(pl.String)
        .str.strip_chars()
        .cast(pl.Float64, strict=False)
    )
    expressions.append(value.alias(col))

expressions.append(
    pl.col("target").cast(pl.Int8)
)

audit_df = (
    train_lf
    .select(expressions)
    .collect()
    .to_pandas()
)

# Flags using training-derived IQR thresholds
outlier_flags = pd.DataFrame(index=audit_df.index)

for col in selected_features:
    lower = limits.loc[col, "lower_limit"]
    upper = limits.loc[col, "upper_limit"]

    outlier_flags[col] = (
        (audit_df[col] < lower)
        | (audit_df[col] > upper)
    ).fillna(False)

# Count unique flagged loans
any_outlier = outlier_flags.any(axis=1)

y = audit_df["target"].astype(int)

print("Original training loans:", len(audit_df))
print("Loans flagged in at least one feature:",
      int(any_outlier.sum()))
print("Flagged percentage:",
      round(100 * any_outlier.mean(), 2), "%")
print("Loans retained after removal:",
      int((~any_outlier).sum()))

print("\nDEFAULT RATE COMPARISON")
comparison = pd.DataFrame({
    "group": [
        "All training loans",
        "Flagged loans",
        "Non-flagged loans",
    ],
    "loans": [
        len(y),
        int(any_outlier.sum()),
        int((~any_outlier).sum()),
    ],
    "default_rate": [
        y.mean(),
        y[any_outlier].mean(),
        y[~any_outlier].mean(),
    ],
})

display(comparison.round(4))

print("\nNUMBER OF OUTLIER FEATURES PER LOAN")
display(
    outlier_flags.sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("features_flagged")
    .reset_index(name="loans")
)

print("\nTRAINING IQR LIMITS")
display(
    limits[
        ["lower_limit", "upper_limit", "flagged_pct"]
    ].round(2)
)

print("\nNo rows removed. Original model unchanged.")


Original training loans: 457355
Loans flagged in at least one feature: 72725
Flagged percentage: 15.9 %
Loans retained after removal: 384630

DEFAULT RATE COMPARISON


,group,loans,default_rate
0,All training loans,457355,0.1393
1,Flagged loans,72725,0.0900
2,Non-flagged loans,384630,0.1486



NUMBER OF OUTLIER FEATURES PER LOAN


,features_flagged,loans
0,0,384630
1,1,42188
2,2,18447
3,3,7184
4,4,3098
5,5,1808



TRAINING IQR LIMITS


,lower_limit,upper_limit,flagged_pct
feature,,,
annual_inc,-23750.0,154250.0,4.61
revol_bal,-13533.5,38382.5,6.08
total_rev_hi_lim,-23750.0,73450.0,6.07
avg_cur_bal,-18815.5,38732.5,5.82
tot_cur_bal,-221130.5,436321.5,4.11



No rows removed. Original model unchanged.



### 06.3 — Experiment A: Original Baseline

We evaluate the existing frozen XGBoost model on the October–December 2015 validation set.

This establishes the reference performance without outlier treatment. We record PR-AUC, ROC-AUC, F1-score, precision, recall, and balanced accuracy.

The model and its classification threshold remain unchanged.

Subsequent experiments will use the same XGBoost configuration and validation observations, modifying only the training-data outlier treatment.


In [3]:

# 06.3 — Original baseline model evaluation

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    accuracy_score,
)

from loanrisk.predict import predict_full

# Load validation data only (2015-10 through 2015-12)
val_df = (
    val_lf
    .select(raw_features + ["target"])
    .collect()
    .to_pandas()
)

X_val_raw = val_df[raw_features].copy()
y_val = val_df["target"].astype(int).to_numpy()

assert len(X_val_raw) == 88810

# Existing frozen model
threshold = float(bundle["threshold"])

predictions = predict_full(X_val_raw, bundle)

p_baseline = predictions[
    "default_risk_score"
].to_numpy(dtype=float)

y_baseline = (p_baseline >= threshold).astype(int)

# Reusable metric function for upcoming experiments
def evaluate_model(name, y_true, probabilities, threshold):
    labels = (probabilities >= threshold).astype(int)

    return {
        "experiment": name,
        "pr_auc": average_precision_score(
            y_true, probabilities
        ),
        "roc_auc": roc_auc_score(
            y_true, probabilities
        ),
        "f1": f1_score(
            y_true, labels, zero_division=0
        ),
        "precision": precision_score(
            y_true, labels, zero_division=0
        ),
        "recall": recall_score(
            y_true, labels, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, labels
        ),
        "accuracy": accuracy_score(
            y_true, labels
        ),
        "flagged_pct": labels.mean() * 100,
    }


baseline_metrics = evaluate_model(
    "A - Original Baseline",
    y_val,
    p_baseline,
    threshold,
)

experiment_results = [baseline_metrics]

print("BASELINE EVALUATION")
print("Validation loans:", len(y_val))
print("Default rate:", round(y_val.mean(), 4))
print("Fixed threshold:", round(threshold, 5))

display(
    pd.DataFrame(experiment_results).round(5)
)

print("\nPreviously recorded validation PR-AUC: 0.29912")
print(
    "Current PR-AUC:",
    round(baseline_metrics["pr_auc"], 5)
)

# Keep baseline metrics for experiment comparison
pd.DataFrame(experiment_results).to_csv(
    REPORT_DIR / "06_experiment_comparison.csv",
    index=False,
)

print("\nBaseline evaluation: COMPLETE")


BASELINE EVALUATION
Validation loans: 88810
Default rate: 0.1484
Fixed threshold: 0.1561


,experiment,pr_auc,roc_auc,f1,precision,recall,balanced_accuracy,accuracy,flagged_pct
0,A - Original Baseline,0.29912,0.71492,0.36443,0.26926,0.56368,0.64851,0.70815,31.07533



Previously recorded validation PR-AUC: 0.29912
Current PR-AUC: 0.29912

Baseline evaluation: COMPLETE



## 06.4 — Experiment B: Outlier Removal

**Objective:** Determine whether removing extreme financial observations improves default-risk prediction.

**Method:**
- Use the five selected financial features.
- Apply IQR limits calculated exclusively from the training period.
- Remove a training loan if at least one selected feature exceeds its limits.
- Retrain XGBoost with unchanged hyperparameters and preprocessing.
- Evaluate on the complete October–December 2015 validation set.

We compare two models trained using the same procedure:

- **A1:** Refit baseline without removing observations.
- **B:** Refit after removing flagged training observations.

**Primary metric:** PR-AUC. Secondary metrics include ROC-AUC, F1, precision and recall.

The 2016 held-out test set remains untouched.


In [5]:

# 06.4 — Controlled outlier-removal experiment

import gc
import time
from sklearn.base import clone
from IPython.display import display

# Retrieve the fitted pipeline from the saved model
pipeline = bundle["pipeline"]

print("Pipeline stages:", list(pipeline.named_steps.keys()))
print("Model:", type(pipeline.steps[-1][1]).__name__)

# Load original training inputs and targets
train_df = (
    train_lf
    .select(raw_features + ["target"])
    .collect()
    .to_pandas()
)

X_train_raw = train_df[raw_features].copy()
y_train = train_df["target"].astype(int).to_numpy()

assert len(X_train_raw) == 457355
assert len(X_val_raw) == 88810

# ---------------------------------------------------
# 1. Recalculate outlier flags on the original rows
# ---------------------------------------------------

remove_mask = np.zeros(len(X_train_raw), dtype=bool)

for col in selected_features:
    values = pd.to_numeric(
        X_train_raw[col], errors="coerce"
    )

    lower = float(limits.loc[col, "lower_limit"])
    upper = float(limits.loc[col, "upper_limit"])

    flags = (
        (values < lower) | (values > upper)
    ).fillna(False).to_numpy()

    remove_mask |= flags

X_filtered = X_train_raw.loc[~remove_mask].copy()
y_filtered = y_train[~remove_mask]

print("OUTLIER REMOVAL SUMMARY")
print("Original training rows:", len(X_train_raw))
print("Removed rows:", int(remove_mask.sum()))
print("Removed percentage:",
      round(remove_mask.mean() * 100, 2), "%")
print("Remaining training rows:", len(X_filtered))
print("Original default rate:", round(y_train.mean(), 4))
print("Filtered default rate:", round(y_filtered.mean(), 4))

assert len(X_filtered) + remove_mask.sum() == len(X_train_raw)
assert len(np.unique(y_filtered)) == 2

# ---------------------------------------------------
# 2. Experiment A1 — Refit original baseline
# ---------------------------------------------------

print("\nTraining A1: Untreated baseline")

start = time.perf_counter()

model_a1 = clone(pipeline)
model_a1.fit(X_train_raw, y_train)

p_a1 = model_a1.predict_proba(X_val_raw)[:, 1]

metrics_a1 = evaluate_model(
    "A1 - Refit Baseline",
    y_val,
    p_a1,
    threshold,
)

print("Training time (seconds):",
      round(time.perf_counter() - start, 1))

# Free fitted model memory
del model_a1
gc.collect()

# ---------------------------------------------------
# 3. Experiment B — Remove outliers
# ---------------------------------------------------

print("\nTraining B: Outlier removal")

start = time.perf_counter()

model_b = clone(pipeline)
model_b.fit(X_filtered, y_filtered)

p_b = model_b.predict_proba(X_val_raw)[:, 1]

metrics_b = evaluate_model(
    "B - IQR Removal",
    y_val,
    p_b,
    threshold,
)

print("Training time (seconds):",
      round(time.perf_counter() - start, 1))

del model_b
gc.collect()

# ---------------------------------------------------
# 4. Compare validation performance
# ---------------------------------------------------

experiment_results = [
    baseline_metrics,
    metrics_a1,
    metrics_b,
]

comparison = pd.DataFrame(experiment_results)

print("\nVALIDATION PERFORMANCE")
display(comparison.round(5))

# Direct difference: B - A1
metrics = [
    "pr_auc",
    "roc_auc",
    "f1",
    "precision",
    "recall",
    "balanced_accuracy",
]

difference = pd.DataFrame({
    "metric": metrics,
    "baseline": [metrics_a1[m] for m in metrics],
    "removal": [metrics_b[m] for m in metrics],
    "difference": [
        metrics_b[m] - metrics_a1[m]
        for m in metrics
    ],
})

print("\nCHANGE AFTER OUTLIER REMOVAL")
display(difference.round(5))

comparison.to_csv(
    REPORT_DIR / "06_experiment_comparison.csv",
    index=False,
)

difference.to_csv(
    REPORT_DIR / "06_removal_impact.csv",
    index=False,
)

print("\nExperiment B complete.")
print("Original saved model remains unchanged.")


Pipeline stages: ['features', 'preprocessing', 'model']
Model: XGBClassifier
OUTLIER REMOVAL SUMMARY
Original training rows: 457355
Removed rows: 72725
Removed percentage: 15.9 %
Remaining training rows: 384630
Original default rate: 0.1393
Filtered default rate: 0.1486

Training A1: Untreated baseline
Training time (seconds): 51.4

Training B: Outlier removal
Training time (seconds): 41.8

VALIDATION PERFORMANCE


,experiment,pr_auc,roc_auc,f1,precision,recall,balanced_accuracy,accuracy,flagged_pct
0,A - Original Baseline,0.29912,0.71492,0.36443,0.26926,0.56368,0.64851,0.70815,31.07533
1,A1 - Refit Baseline,0.29912,0.71492,0.36443,0.26926,0.56368,0.64851,0.70815,31.07533
2,B - IQR Removal,0.29846,0.71409,0.36342,0.26869,0.56133,0.64750,0.70810,31.01115



CHANGE AFTER OUTLIER REMOVAL


,metric,baseline,removal,difference
0,pr_auc,0.29912,0.29846,-0.00066
1,roc_auc,0.71492,0.71409,-0.00083
2,f1,0.36443,0.36342,-0.00101
3,precision,0.26926,0.26869,-0.00057
4,recall,0.56368,0.56133,-0.00235
5,balanced_accuracy,0.64851,0.64750,-0.00100



Experiment B complete.
Original saved model remains unchanged.


### removing outliers doent affect our model much and xgboost is good at handling numerical outliers thus we do not remove outliers.
